# Block 1 — Tracker State and Straight-Line Projection

The AMS-02 silicon tracker reconstructs the trajectory of a charged particle before it reaches the Electromagnetic Calorimeter (ECAL). To analyze the energy deposited in the ECAL relative to the incoming particle, we must project the reconstructed track to the center of each ECAL layer.

In this notebook, we will:

1. represent a reconstructed track using a reference position and two direction angles;
2. derive the equations for straight-line propagation;
3. project an example track to the 18 ECAL layer centers;
4. verify the special case of a vertical track.

This block works entirely in continuous spatial coordinates. It does **not** yet assign projected positions to ECAL cells or account for alternating readout directions. Those operations belong to Block 2.

## 1. Coordinate and angle conventions

We use the local ECAL coordinate system established in Block 0:

- $x$ and $y$ describe transverse positions across the ECAL face;
- $z$ increases through the ECAL depth;
- all positions and distances are measured in millimetres;
- all angles are measured in radians in the implementation.

A reconstructed track is defined at a reference position

$$
\mathbf{r}_0 =
\begin{pmatrix}
x_0 \\
y_0 \\
z_0
\end{pmatrix}.
$$

Its direction is described by:

- $\theta$: the polar angle measured from the positive $z$-axis;
- $\phi$: the azimuthal angle measured from the positive $x$-axis toward the positive $y$-axis.

The corresponding unit direction vector is

$$
\hat{\mathbf{u}}
=
\begin{pmatrix}
\sin\theta\cos\phi \\
\sin\theta\sin\phi \\
\cos\theta
\end{pmatrix}.
$$

Therefore, the track can be written parametrically as

$$
\mathbf{r}(s)=\mathbf{r}_0+s\hat{\mathbf{u}},
$$

where $s$ is the signed distance travelled along the track.

## 2. Projection to a known z-coordinate

Suppose an ECAL layer is centered at $z=z_\ell$. From the $z$-component of the parametric track equation,

$$
z_\ell=z_0+s_\ell\cos\theta.
$$

Solving for the distance parameter gives

$$
s_\ell=\frac{z_\ell-z_0}{\cos\theta}.
$$

Substituting this result into the $x$- and $y$-components gives

$$
x_\ell
=
x_0+(z_\ell-z_0)\tan\theta\cos\phi,
$$

and

$$
y_\ell
=
y_0+(z_\ell-z_0)\tan\theta\sin\phi.
$$

It is useful to define the transverse slopes with respect to $z$:

$$
t_x=\frac{dx}{dz}=\tan\theta\cos\phi,
$$

$$
t_y=\frac{dy}{dz}=\tan\theta\sin\phi.
$$

The projection equations then become

$$
x_\ell=x_0+(z_\ell-z_0)t_x,
$$

$$
y_\ell=y_0+(z_\ell-z_0)t_y.
$$

These equations assume that the reconstructed trajectory can be propagated as a straight line between the reference point and the ECAL. Effects such as multiple scattering, magnetic curvature, track-fit uncertainty, and shower development are outside the scope of this block.

In [3]:
from math import cos, radians, sin, tan
from pathlib import Path

from ams_ecal import load_geometry


def find_repository_root(start: Path) -> Path:
    """Find the repository root from Jupyter's current directory."""

    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate

    raise FileNotFoundError("Could not locate the repository root.")


repository_root = find_repository_root(Path.cwd())
geometry = load_geometry(repository_root / "configs" / "geometry.yaml")

len(geometry.uniform_layer_centers_z_mm)

18

## 3. Numerical projection example

We now define an example reconstructed track with reference position

$$
(x_0,y_0,z_0)=(10,-20,0)\ \text{mm},
$$

and direction angles

$$
\theta=10^\circ,
\qquad
\phi=30^\circ.
$$

Although the angles are easier to describe in degrees, Python's trigonometric functions expect radians. We therefore convert them using `radians()`.

The transverse slopes $t_x$ and $t_y$ describe how much the track's $x$- and $y$-coordinates change for every millimetre travelled in the positive $z$-direction.

In [4]:
# Example reconstructed track state in the local ECAL coordinate system.
x0_mm = 10.0
y0_mm = -20.0
z0_mm = 0.0

theta_rad = radians(10.0)
phi_rad = radians(30.0)

slope_x = tan(theta_rad) * cos(phi_rad)
slope_y = tan(theta_rad) * sin(phi_rad)

slope_x, slope_y

(0.1527036446661393, 0.08816349035423247)

Because both slopes are positive, the projected $x$- and $y$-coordinates should increase as the track passes deeper into the ECAL.

The reference value $y_0=-20\ \text{mm}$ is negative, so an increasing $y$-coordinate means that it becomes less negative before potentially crossing zero.

We will evaluate the projection equations at every layer-center $z$-coordinate supplied by the Block 0 geometry model.

In [5]:
projected_points = tuple(
    (
        x0_mm + (z_mm - z0_mm) * slope_x,
        y0_mm + (z_mm - z0_mm) * slope_y,
        z_mm,
    )
    for z_mm in geometry.uniform_layer_centers_z_mm
)

projected_points[0], projected_points[-1]

((10.706254356580894, -19.592243857111676, 4.625),
 (34.7189024803313, -5.728534998908618, 161.875))

## 4. Interpreting the result

The first projected point lies at the center of the first ECAL layer, while the last lies at the center of the eighteenth layer.

For this example:

- the $x$-coordinate increases from approximately $10.71\ \text{mm}$ to $34.72\ \text{mm}$;
- the $y$-coordinate increases from approximately $-19.59\ \text{mm}$ to $-5.73\ \text{mm}$;
- the $z$-coordinates come directly from the detector geometry.

The projection currently returns continuous physical coordinates. For example, $x=10.71\ \text{mm}$ has not yet been converted into an integer ECAL cell index.

Keeping continuous projection and discrete cell mapping separate lets us test the track geometry independently of the detector readout mapping.

## 5. Vertical-track limit

A useful physical and numerical check is the vertical-track case.

If

$$
\theta=0,
$$

then

$$
\tan\theta=0.
$$

Consequently,

$$
t_x=t_y=0,
$$

regardless of the value of $\phi$. The projection equations reduce to

$$
x_\ell=x_0,
\qquad
y_\ell=y_0.
$$

A vertical track should therefore retain the same transverse coordinates at every ECAL layer.

In [6]:
vertical_theta_rad = 0.0

vertical_points = tuple(
    (
        x0_mm
        + (z_mm - z0_mm)
        * tan(vertical_theta_rad)
        * cos(phi_rad),
        y0_mm
        + (z_mm - z0_mm)
        * tan(vertical_theta_rad)
        * sin(phi_rad),
        z_mm,
    )
    for z_mm in geometry.uniform_layer_centers_z_mm
)

assert all(x_mm == x0_mm for x_mm, _, _ in vertical_points)
assert all(y_mm == y0_mm for _, y_mm, _ in vertical_points)

vertical_points[0], vertical_points[-1]

((10.0, -20.0, 4.625), (10.0, -20.0, 161.875))

## Checkpoint conclusions

We have derived and numerically verified the straight-line track-projection model:

$$
x(z)=x_0+(z-z_0)\tan\theta\cos\phi,
$$

$$
y(z)=y_0+(z-z_0)\tan\theta\sin\phi.
$$

The numerical example showed that an inclined track changes its transverse position as it travels through the ECAL. The vertical-track test confirmed that setting $\theta=0$ preserves $x_0$ and $y_0$ at every layer.

At this checkpoint, the equations exist only as exploratory notebook code. The next step is to move the physical track state into an immutable `TrackState` class and move the projection calculation into reusable, testable functions.

Cell indexing, alternating ECAL readout views, and shower-centered cropping remain outside Block 1.